# Module 01 — Lecture 3: First CUDA Programs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/GPU-Programming-For-Computational-Neuroscience/blob/main/module_01_cuda_fundamentals/03_first_cuda_programs.ipynb)

---

Time to write real code. In this lecture we build and run two complete CUDA programs:

1. **`hello_cuda`** — prints thread identities; your GPU speaking to you
2. **`vector_add`** — GPU vs CPU benchmark with CUDA event timing

We end by visualizing GPU bandwidth as a function of array size — a skill you will use throughout the course to diagnose performance.

**Learning objectives:**
- Compile CUDA code with `nvcc` inside a notebook
- Use `cudaEventRecord` and `cudaEventElapsedTime` for precise timing
- Measure and interpret effective memory bandwidth
- Confirm GPU correctness by comparing against a CPU reference

In [ ]:
!nvidia-smi

## Program 1: Hello CUDA

Every thread announces itself. This sounds trivial, but it confirms:
- The kernel runs on the GPU
- Each thread has a unique identity
- `printf` from GPU works (via hardware buffer)
- `cudaDeviceSynchronize` flushes GPU output to stdout

In [ ]:
%%writefile hello_cuda.cu
#include <stdio.h>
#include <cuda_runtime.h>

#define CUDA_CHECK(call) do {                                        \
    cudaError_t e = (call);                                          \
    if (e != cudaSuccess) {                                          \
        fprintf(stderr, "CUDA error at %s:%d: %s\n",                \
                __FILE__, __LINE__, cudaGetErrorString(e));          \
        exit(1); }                                                   \
} while(0)

// Each thread prints its location in the hierarchy
__global__ void hello_kernel() {
    int global_id = blockIdx.x * blockDim.x + threadIdx.x;
    printf("  Block %2d | Thread %2d | Global ID = %3d\n",
           blockIdx.x, threadIdx.x, global_id);
}

int main() {
    // Query GPU
    cudaDeviceProp prop;
    CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));
    printf("Running on: %s\n", prop.name);
    printf("Warp size:  %d\n\n", prop.warpSize);

    int num_blocks = 3, threads = 8;
    printf("Launching %d blocks x %d threads = %d total threads:\n",
           num_blocks, threads, num_blocks * threads);

    hello_kernel<<<num_blocks, threads>>>();
    CUDA_CHECK(cudaDeviceSynchronize());  // flush GPU printf buffer

    printf("\nAll threads complete.\n");
    return 0;
}

In [ ]:
!nvcc -O2 -o hello_cuda hello_cuda.cu && ./hello_cuda

**Output analysis:**

Notice the threads within each block are grouped, but the order between blocks may vary — blocks can execute in any order on the GPU. This is expected and correct. The GPU scheduler assigns blocks to available SMs as they become free.

> **Important:** Never write code that assumes blocks execute in a specific order.

## Program 2: Vector Addition with Timing

A classic first GPU program: `C[i] = A[i] + B[i]` for N elements in parallel.

This is a **memory-bound** kernel — the bottleneck is reading A and B from global memory, not the addition itself. It lets us directly measure **memory bandwidth**, which is the key metric for most neuroscience simulation kernels.

In [ ]:
%%writefile vector_add.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>

#define CUDA_CHECK(call) do {                                        \
    cudaError_t e = (call);                                          \
    if (e != cudaSuccess) {                                          \
        fprintf(stderr, "CUDA error: %s\n", cudaGetErrorString(e)); \
        exit(1); }                                                   \
} while(0)

// GPU kernel: each thread adds one element pair
__global__ void vadd_kernel(const float* A, const float* B, float* C, int N) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N) C[i] = A[i] + B[i];
}

int main(int argc, char** argv) {
    // Array size from command line, or default 1M elements
    int N = (argc > 1) ? atoi(argv[1]) : 1 << 20;
    size_t bytes = N * sizeof(float);

    // Host arrays
    float* h_A = (float*)malloc(bytes);
    float* h_B = (float*)malloc(bytes);
    float* h_C = (float*)malloc(bytes);
    for (int i = 0; i < N; i++) { h_A[i] = 1.0f; h_B[i] = 2.0f; }

    // Device arrays
    float *d_A, *d_B, *d_C;
    CUDA_CHECK(cudaMalloc(&d_A, bytes));
    CUDA_CHECK(cudaMalloc(&d_B, bytes));
    CUDA_CHECK(cudaMalloc(&d_C, bytes));
    CUDA_CHECK(cudaMemcpy(d_A, h_A, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_B, h_B, bytes, cudaMemcpyHostToDevice));

    // Time GPU kernel with CUDA events (microsecond precision)
    cudaEvent_t t0, t1;
    CUDA_CHECK(cudaEventCreate(&t0));
    CUDA_CHECK(cudaEventCreate(&t1));

    int threads = 256;
    int blocks  = (N + threads - 1) / threads;

    CUDA_CHECK(cudaEventRecord(t0));
    vadd_kernel<<<blocks, threads>>>(d_A, d_B, d_C, N);
    CUDA_CHECK(cudaEventRecord(t1));
    CUDA_CHECK(cudaEventSynchronize(t1));

    float ms;
    CUDA_CHECK(cudaEventElapsedTime(&ms, t0, t1));

    CUDA_CHECK(cudaMemcpy(h_C, d_C, bytes, cudaMemcpyDeviceToHost));

    // Verify
    int ok = 1;
    for (int i = 0; i < N; i++) if (fabsf(h_C[i] - 3.0f) > 1e-5f) { ok = 0; break; }

    // Effective bandwidth: reads A + B, writes C
    double bw = 3.0 * bytes / (ms * 1e-3) / 1e9;

    printf("N=%-12d  time=%.3f ms  bandwidth=%.1f GB/s  %s\n",
           N, ms, bw, ok ? "PASS" : "FAIL");

    CUDA_CHECK(cudaEventDestroy(t0)); CUDA_CHECK(cudaEventDestroy(t1));
    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    free(h_A); free(h_B); free(h_C);
    return 0;
}

In [ ]:
!nvcc -O2 -o vector_add vector_add.cu -lm

In [ ]:
# Run with different array sizes to see bandwidth scaling
import subprocess
import numpy as np
import matplotlib.pyplot as plt

sizes = [2**k for k in range(10, 26)]   # 1K to 32M elements
bandwidths = []
times_ms = []

for N in sizes:
    result = subprocess.run(['./vector_add', str(N)], capture_output=True, text=True)
    line = result.stdout.strip()
    print(line)
    parts = line.split()
    times_ms.append(float(parts[1].replace('time=','').replace('ms','')))
    bandwidths.append(float(parts[2].replace('bandwidth=','').replace('GB/s','')))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

sizes_mb = [N * 4 / 1e6 for N in sizes]   # size in MB (float = 4 bytes)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Bandwidth vs array size
ax1.semilogx([s * 4 / 1e6 for s in sizes], bandwidths, 'b-o', markersize=5)
ax1.axhline(y=max(bandwidths), color='r', linestyle='--', alpha=0.7,
            label=f'Peak: {max(bandwidths):.0f} GB/s')
ax1.set_xlabel('Array size per buffer (MB)', fontsize=12)
ax1.set_ylabel('Effective Bandwidth (GB/s)', fontsize=12)
ax1.set_title('GPU Memory Bandwidth vs Problem Size', fontsize=13)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)
ax1.set_ylim(0, max(bandwidths) * 1.15)

# Kernel time vs array size
ax2.loglog(sizes, times_ms, 'g-o', markersize=5)
ax2.set_xlabel('Number of elements (N)', fontsize=12)
ax2.set_ylabel('Kernel time (ms)', fontsize=12)
ax2.set_title('Kernel Execution Time vs N', fontsize=13)
ax2.grid(True, alpha=0.3)

# Annotate neuron simulation scale
ax2.axvline(10000, color='orange', linestyle='--', alpha=0.8, label='10,000 neurons')
ax2.axvline(100000, color='red', linestyle='--', alpha=0.8, label='100,000 neurons')
ax2.legend(fontsize=10)

plt.tight_layout()
plt.savefig('bandwidth_profile.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nPeak measured bandwidth: {max(bandwidths):.1f} GB/s")
print(f"Time for 10,000-neuron step: ~{times_ms[7]:.3f} ms")
print(f"  → Can simulate {1000/times_ms[7]:.0f} real-time ms per second")

## Reading the Bandwidth Plot

**Small arrays (< 1 MB):** Bandwidth is low — the kernel launch overhead (~5–10 μs) dominates over actual computation time.

**Large arrays (> 10 MB):** Bandwidth plateaus near the GPU's theoretical limit (T4: ~300 GB/s). The GPU is fully saturated.

**Sweet spot for neuroscience:** At N = 10,000–100,000 neurons, a single float state update takes < 0.1 ms. The simulation is fast. For very small N (<1,000), the GPU is overkill — CPU would be faster due to lower launch overhead.

This is your first performance diagnostic tool. Whenever you write a kernel, ask:
- What fraction of peak bandwidth am I achieving?
- Am I memory-bound or compute-bound?

## Compile Flags Reference

```bash
nvcc -O2 -o program file.cu           # Basic optimized compile
nvcc -O2 -arch=sm_75 -o prog file.cu  # Target specific GPU arch (T4 = sm_75)
nvcc -O2 -lineinfo -o prog file.cu    # Keep line info for profiler
nvcc -O2 -lm -o prog file.cu          # Link math library
nvcc -O2 -lcublas -o prog file.cu     # Link cuBLAS (Module 06)

# Architectures by GPU:
#   T4      → sm_75
#   V100    → sm_70
#   A100    → sm_80
#   RTX 3090 → sm_86
#   RTX 4090 → sm_89
```

## Summary

You have now written, compiled, and run real CUDA programs.

| Skill | Demonstrated |
|-------|--------------|
| `%%writefile` + `!nvcc` | Compile CUDA in Colab notebooks |
| Thread indexing | `blockIdx.x * blockDim.x + threadIdx.x` |
| Memory management | `cudaMalloc`, `cudaMemcpy`, `cudaFree` |
| CUDA event timing | `cudaEventRecord`, `cudaEventElapsedTime` |
| Bandwidth analysis | Effective bandwidth = bytes transferred / time |
| Performance scale | GPU excels at N > 10,000 for memory-bound kernels |

**Proceed to:** [Exercise 01](exercises/ex01_stub.ipynb) — write your own parallel kernel from scratch.